# Лабораторна робота: Аналіз даних Titanic: Класифікація виживання пасажирів

Мета роботи побудувати та порівняти моделі машинного навчання (Логістична регресія, Дерево рішень, Випадковий ліс) для передбачення виживання пасажирів Титаніка.

П.с. Для написання коментарів було використано ШІ


**Етапи:**
1. Підготовка даних та EDA (Розвідувальний аналіз).
2. Feature Engineering (Створення нових ознак).
3. Побудова та оцінка базових моделей.
4. Оптимізація гіперпараметрів.
5. Аналіз важливості ознак.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report

sns.set(style="whitegrid")
print("Бібліотеки успішно імпортовано!")

In [ ]:
# Завантаження вбудованого набору даних з Seaborn
df = sns.load_dataset('titanic')

print("Перші 10 рядків")
display(df.head(10))

print("\n Інформація про дані та пропуски")
print(df.info())

print("\n Базова статистика")
display(df.describe())

plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title("Мапа пропущених значень")
plt.show()

In [ ]:
# 1. Обробка пропущених значень
df['age'] = df['age'].fillna(df['age'].mean())


most_common_embarked = df['embarked'].mode()[0]
df['embarked'] = df['embarked'].fillna(most_common_embarked)

# Також видаляємо дублюючі або технічні колонки з датасету seaborn ('embark_town', 'alive', 'who', 'class', 'adult_male')
cols_to_drop = ['deck', 'embark_town', 'alive', 'who', 'class', 'adult_male']
df = df.drop(columns=cols_to_drop)

# 2. Створення нових ознак (Feature Engineering)
# FamilySize = SibSp (брати/сестри/чоловік/дружина) + Parch (батьки/діти)
df['FamilySize'] = df['sibsp'] + df['parch']

# 3. Кодування категоріальних змінних (One-Hot Encoding)
# get_dummies перетворить 'sex' та 'embarked' у числові колонки (0 та 1)
# drop_first=True видаляє першу колонку, щоб уникнути мультиколінеарності (пастки фіктивних змінних)
df = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True)

print("Дані після обробки:")
display(df.head())

In [ ]:
X = df.drop('survived', axis=1)
y = df['survived']

# Поділ на тренувальну (80%) та тестову (20%) вибірки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Розмір тренувальної вибірки: {X_train.shape}")
print(f"Розмір тестової вибірки: {X_test.shape}")

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results_table = pd.DataFrame(columns=["Model", "Accuracy", "Precision", "Recall", "F1-Score"])

plt.figure(figsize=(10, 8))

for name, model in models.items():
    # 1. Тренування
    model.fit(X_train, y_train)
    
    # 2. Передбачення
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] # Ймовірності для ROC-кривої
    
    # 3. Метрики
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Додавання в таблицю (використовуємо concat замість append)
    new_row = pd.DataFrame({
        "Model": [name], "Accuracy": [acc], "Precision": [prec], 
        "Recall": [rec], "F1-Score": [f1]
    })
    results_table = pd.concat([results_table, new_row], ignore_index=True)
    
    # 4. Матриця плутанини (окремо виводимо графік для кожної - тут для прикладу текстово, 
    # але можна розкоментувати sns.heatmap нижче для кожної моделі)
    print(f"\n--- {name} Confusion Matrix ---")
    print(confusion_matrix(y_test, y_pred))
    
    # 5. ROC-крива (додаємо на загальний графік)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2) # Діагональ випадкового вгадування
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc="lower right")
plt.show()

print("\n--- Порівняльна таблиця метрик ---")
display(results_table)

In [ ]:
print("--- Крос-валідація (Cross-Validation) ---")
# Оцінка стабільності моделей на 5 фолдах
for name, model in models.items():
    if name != "Random Forest": # Random Forest оптимізуємо окремо
        cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
        print(f"{name} CV Average Accuracy: {cv_scores.mean():.4f}")

print("\n--- Grid Search для Random Forest ---")
# Визначаємо сітку параметрів
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print(f"Найкращі параметри: {grid_search.best_params_}")
print(f"Найкраща точність на валідації: {grid_search.best_score_:.4f}")

In [ ]:
# Аналіз важливості ознак для найкращої моделі Random Forest
importances = best_rf.feature_importances_
feature_names = X.columns
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df, palette='viridis')
plt.title('Важливість ознак у моделі Random Forest')
plt.xlabel('Коефіцієнт важливості')
plt.ylabel('Ознака')
plt.show()

print("""
--- Висновки та Ідеї ---
1. Найвпливовіші ознаки: Зазвичай це стать (sex_male) та ціна квитка (fare) або вік.
2. Пропозиція покращення: 
   - Можна виділити титули з імен (Mr, Mrs, Miss), оскільки вони краще вказують на соціальний статус, ніж просто Pclass.
   - Створити ознаку IsAlone (якщо FamilySize == 0).
   - Використати метод KNN Imputer для більш точного заповнення віку замість середнього значення.
""")